# 🤖 RAG PDF Chatbot - Google Colab Setup

This notebook sets up and runs a production-ready RAG (Retrieval-Augmented Generation) chatbot using Streamlit.

**Features:**
- 📄 Multi-PDF support with intelligent chunking
- 🔍 Semantic similarity search using embeddings
- 🧠 Multiple LLM options (OpenAI, Ollama, HuggingFace)
- 🚀 Production-ready with caching and optimization
- 💬 Interactive Streamlit interface

## Step 1: Install Dependencies

In [ ]:
# Install essential packages
!pip install --upgrade pip setuptools wheel

# Install core dependencies
!pip install -q langchain langchain-openai langchain-community
!pip install -q streamlit
!pip install -q PyPDF2 faiss-cpu chromadb sentence-transformers
!pip install -q transformers torch torchvision
!pip install -q huggingface-hub openai
!pip install -q pydantic python-dotenv

print("✅ All dependencies installed successfully!")

## Step 2: Download Code Files

Create the main application files

In [ ]:
import os
from pathlib import Path

# Create project directory
project_dir = "/content/rag_chatbot"
os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)

print(f"✅ Project directory created: {project_dir}")

## Step 3: Create RAG Pipeline Module

In [ ]:
# Create rag_pipeline.py
rag_pipeline_code = '''
"""RAG Pipeline Implementation"""
import os
from typing import List, Tuple, Optional
from pathlib import Path
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate

class RAGPipeline:
    def __init__(self, model_name="gpt-3.5-turbo", embedding_model="sentence-transformers/all-MiniLM-L6-v2",
                 provider="openai", api_key=None, chunk_size=512, chunk_overlap=50, top_k=3):
        self.model_name = model_name
        self.embedding_model = embedding_model
        self.provider = provider
        self.api_key = api_key
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.top_k = top_k
        
        self.embeddings = None
        self.vector_store = None
        self.llm = None
        self.retriever = None
        self.qa_chain = None
        
        self._initialize_embeddings()
        self._initialize_llm()
    
    def _initialize_embeddings(self):
        self.embeddings = HuggingFaceEmbeddings(model_name=self.embedding_model)
        print(f"✅ Embeddings initialized: {self.embedding_model}")
    
    def _initialize_llm(self):
        if self.provider == "openai":
            self.llm = OpenAI(openai_api_key=self.api_key, model_name=self.model_name, temperature=0.7)
            print(f"✅ OpenAI LLM initialized: {self.model_name}")
    
    def build_vector_store(self, pdf_paths: List[str]):
        documents = []
        for pdf_path in pdf_paths:
            loader = PyPDFLoader(pdf_path)
            docs = loader.load()
            documents.extend(docs)
        
        splitter = RecursiveCharacterTextSplitter(chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)
        chunks = splitter.split_documents(documents)
        
        self.vector_store = FAISS.from_documents(chunks, self.embeddings)
        self.retriever = self.vector_store.as_retriever(search_type="similarity", search_kwargs={"k": self.top_k})
        self._create_qa_chain()
        print(f"✅ Vector store built with {len(chunks)} chunks")
    
    def _create_qa_chain(self):
        prompt = PromptTemplate(template="Use context to answer: {context}\\nQuestion: {question}\\nAnswer:",
                              input_variables=["context", "question"])
        self.qa_chain = RetrievalQA.from_chain_type(llm=self.llm, chain_type="stuff",
                                                    retriever=self.retriever, return_source_documents=True)
    
    def query(self, question: str, temperature: float = 0.7) -> Tuple[str, List]:
        response = self.qa_chain({"query": question})
        answer = response.get("result", "")
        sources = [(doc.metadata.get("source"), doc.page_content[:200]) for doc in response.get("source_documents", [])]
        return answer, sources
'''

with open("rag_pipeline.py", "w") as f:
    f.write(rag_pipeline_code)

print("✅ Created rag_pipeline.py")

## Step 4: Create Streamlit App

In [ ]:
# Create streamlit_app.py
streamlit_code = '''
import streamlit as st
import os
import tempfile
from rag_pipeline import RAGPipeline

st.set_page_config(page_title="RAG PDF Chatbot", page_icon="🤖", layout="wide")
st.title("🤖 RAG PDF Chatbot")
st.markdown("Ask questions about your uploaded PDFs")

# Initialize session state
if "rag_pipeline" not in st.session_state:
    st.session_state.rag_pipeline = None
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "vector_store_ready" not in st.session_state:
    st.session_state.vector_store_ready = False

# Sidebar
with st.sidebar:
    st.header("⚙️ Configuration")
    api_key = st.text_input("OpenAI API Key", type="password")
    chunk_size = st.slider("Chunk Size", 256, 1024, 512)
    top_k = st.slider("Retrieve Top-K", 1, 10, 3)
    temperature = st.slider("Temperature", 0.0, 1.0, 0.7)
    
    st.subheader("📁 Upload PDFs")
    uploaded_files = st.file_uploader("Upload PDF files", type="pdf", accept_multiple_files=True)
    
    if uploaded_files and st.button("🚀 Process PDFs"):
        with st.spinner("Processing..."):
            st.session_state.rag_pipeline = RAGPipeline(
                api_key=api_key, chunk_size=chunk_size, top_k=top_k
            )
            
            pdf_paths = []
            with tempfile.TemporaryDirectory() as tmpdir:
                for file in uploaded_files:
                    path = os.path.join(tmpdir, file.name)
                    with open(path, "wb") as f:
                        f.write(file.getbuffer())
                    pdf_paths.append(path)
                
                st.session_state.rag_pipeline.build_vector_store(pdf_paths)
            
            st.session_state.vector_store_ready = True
            st.success("✅ PDFs processed!")

# Main chat interface
if not st.session_state.vector_store_ready:
    st.warning("⬅️ Upload PDFs to get started")
else:
    # Display chat history
    for msg in st.session_state.chat_history:
        if msg["role"] == "user":
            st.write(f"**You:** {msg['content']}")
        else:
            st.write(f"**Assistant:** {msg['content']}")
    
    # Chat input
    col1, col2 = st.columns([0.85, 0.15])
    with col1:
        user_input = st.text_input("Ask a question:")
    with col2:
        send = st.button("Send")
    
    if send and user_input:
        st.session_state.chat_history.append({"role": "user", "content": user_input})
        
        with st.spinner("Generating response..."):
            answer, sources = st.session_state.rag_pipeline.query(user_input, temperature)
            st.session_state.chat_history.append({"role": "assistant", "content": answer})
            st.write(f"**Assistant:** {answer}")
            
            if sources:
                st.markdown("### 📚 Sources")
                for src, content in sources:
                    st.write(f"*{src}*: {content}...")
'''

with open("streamlit_app.py", "w") as f:
    f.write(streamlit_code)

print("✅ Created streamlit_app.py")

## Step 5: Launch Streamlit App

**Option A: Using Streamlit Cloud (Recommended)**

1. Push your code to GitHub
2. Go to https://streamlit.io/cloud
3. Deploy directly from your GitHub repo

**Option B: Using Colab with Ngrok (This Cell)**

In [ ]:
# Install ngrok for exposing Streamlit
!pip install -q pyngrok

# Get ngrok token (optional but recommended)
# You can get a free token from https://dashboard.ngrok.com/
# ngrok_token = "your_ngrok_token_here"

print("✅ Ngrok installed. Ready to launch Streamlit!")

In [ ]:
# Launch Streamlit with ngrok tunnel
from pyngrok import ngrok
import subprocess
import time

# Set ngrok token if you have one
# ngrok.set_auth_token("your_token")

# Start Streamlit in background
!streamlit run streamlit_app.py --server.headless true --server.port 8501 > /tmp/streamlit.log 2>&1 &

time.sleep(3)  # Wait for Streamlit to start

# Create ngrok tunnel
public_url = ngrok.connect(8501, "http")
print(f"🌐 Your Streamlit app is live at: {public_url}")
print("\n📌 Share this URL to access your RAG chatbot!")

## Step 6: Configuration Tips

### API Keys Setup
```python
# Create a .env file with your API keys
OPENAI_API_KEY=sk-...
HUGGINGFACE_API_KEY=hf_...
```

### Deployment Options

1. **Streamlit Cloud** (Free, easiest)
   - Push to GitHub
   - Deploy from Streamlit Cloud

2. **Heroku** ($7/month)
   ```bash
   heroku create your-app-name
   git push heroku main
   ```

3. **AWS/GCP/Azure**
   - Docker container deployment
   - See deployment guide below

4. **RunPod/Vast.ai** (GPU deployment)
   - Rent GPU for production inference

## Step 7: Monitor Logs

In [ ]:
# View Streamlit logs
!tail -f /tmp/streamlit.log

## References

- [Streamlit Documentation](https://docs.streamlit.io)
- [LangChain Documentation](https://python.langchain.com)
- [OpenAI API Documentation](https://platform.openai.com/docs)
- [FAISS Vector Database](https://github.com/facebookresearch/faiss)
- [HuggingFace Models](https://huggingface.co/models)